# CardioScore full public validation campaign — v4

This version uses Dryad's **authenticated API download path**. Dryad now requires an API token for file bytes. The notebook exchanges your API account ID/secret for a short-lived token automatically; the token is never printed or saved in the results.

### One-time setup before Run all
1. Sign in to Dryad with ORCID.
2. On **My account**, create a **Dryad API account**.
3. In Colab, click the **key / Secrets** icon in the left sidebar.
4. Add secrets named exactly `DRYAD_ID` and `DRYAD_SECRET` and enable notebook access.

Do **not** paste either secret into notebook source or chat.

**Runtime:** CPU is sufficient. The notebook reuses `MyDrive/CardioScore_validation_campaign_v1`, so completed downloads/results are not repeated.


In [ ]:
#@title 1) Verify Dryad secrets are available
from google.colab import userdata
missing = []
for key in ('DRYAD_ID', 'DRYAD_SECRET'):
    try:
        value = userdata.get(key)
        if not value:
            missing.append(key)
    except Exception:
        missing.append(key)
if missing:
    raise RuntimeError(
        'Missing Colab secret(s): ' + ', '.join(missing) +
        '. Add them using the key/Secrets icon in the left sidebar and enable notebook access.'
    )
print('✓ Dryad credentials are available to this notebook (values hidden).')


In [ ]:
#@title 2) Run SHA-pinned authenticated campaign
import hashlib, urllib.request, os
os.environ['CARDIOSCORE_CAMPAIGN_AUTO_DOWNLOAD'] = '1'
RUNNER_SHA = 'a46f27162a70bd100d39faed1c1c59e4b217c884'
RUNNER_URL = 'https://raw.githubusercontent.com/Virelion-Biotech/Virelion-CardioScore/a46f27162a70bd100d39faed1c1c59e4b217c884/scripts/validation/run_validation_campaign_colab_v4.py'
source = urllib.request.urlopen(RUNNER_URL).read()
print('Runner commit:', RUNNER_SHA)
print('Runner SHA-256:', hashlib.sha256(source).hexdigest())
exec(compile(source, 'run_validation_campaign_colab_v4.py', 'exec'), {'__name__': '__main__'})


## Expected behavior

- Preflight, Blinova, PyMEA, SiMulTox, Geiger and Altrocchi should resume from Drive.
- Ballouz/Vandenberg and Daley should download through the authenticated Dryad API, verify the published file sizes/SHA-256 digests, then run inventory/analysis.
- Patel is retried with authentication. If the historical DOI still exposes no current record/files, it remains explicitly blocked rather than being substituted.
- The final `CardioScore_public_validation_campaign_results.zip` is saved to Drive and downloaded automatically.
